In [1]:
import os
import numpy as np
import librosa

AUDIO_DIR = "all_audio"
SAMPLE_RATE = 16000
N_MFCC = 40

In [4]:
TARGET_DURATION = 3
TARGET_LENGTH = int(SAMPLE_RATE * TARGET_DURATION)

y, sr = librosa.load(path, sr=SAMPLE_RATE)

if len(y) < TARGET_LENGTH:
    y = np.pad(y, (0, TARGET_LENGTH - len(y)))
else:
    y = y[:TARGET_LENGTH]

print("Duration (sec):", len(y)/sr)

Duration (sec): 3.0


In [5]:
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
delta = librosa.feature.delta(mfcc)
delta2 = librosa.feature.delta(mfcc, order=2)

features = np.vstack([mfcc, delta, delta2])

print("MFCC shape:", mfcc.shape)
print("Final feature shape:", features.shape)

MFCC shape: (40, 94)
Final feature shape: (120, 94)


In [6]:
lengths = []

files = os.listdir(AUDIO_DIR)[:50]

for file in files:
    path = os.path.join(AUDIO_DIR, file)

    y, sr = librosa.load(path, sr=SAMPLE_RATE)
    y = librosa.util.fix_length(y, size=int(SAMPLE_RATE * 3))

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    features = np.vstack([mfcc, delta, delta2])
    lengths.append(features.shape[1])

print("Unique lengths:", set(lengths))

Unique lengths: {94}


In [7]:
SAVE_DIR = "features_mfcc"
os.makedirs(SAVE_DIR, exist_ok=True)

for file in os.listdir(AUDIO_DIR):
    if file.endswith(".wav"):
        path = os.path.join(AUDIO_DIR, file)

        y, sr = librosa.load(path, sr=SAMPLE_RATE)
        y = librosa.util.fix_length(y, size=int(SAMPLE_RATE * 3))

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)

        features = np.vstack([mfcc, delta, delta2])

        # normalize
        features = (features - np.mean(features)) / np.std(features)

        # only use if needed
        # features = pad_or_trim(features)

        save_path = os.path.join(SAVE_DIR, file.replace(".wav", ".npy"))
        np.save(save_path, features)

print("Done.")

Done.


In [8]:
sample = os.listdir(SAVE_DIR)[0]
data = np.load(os.path.join(SAVE_DIR, sample))

print("Saved feature shape:", data.shape)

Saved feature shape: (120, 94)


Now for Mell Spectrogram

In [9]:
N_MELS = 128

In [10]:
path = os.path.join(AUDIO_DIR, os.listdir(AUDIO_DIR)[0])

y, sr = librosa.load(path, sr=SAMPLE_RATE)
y = librosa.util.fix_length(y, size=int(SAMPLE_RATE * 3))

mel = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_mels=N_MELS
)

# convert to log scale (critical)
mel_db = librosa.power_to_db(mel, ref=np.max)

print("Mel shape:", mel_db.shape)

Mel shape: (128, 94)


In [11]:
lengths = []

files = os.listdir(AUDIO_DIR)[:50]

for file in files:
    path = os.path.join(AUDIO_DIR, file)

    y, sr = librosa.load(path, sr=SAMPLE_RATE)
    y = librosa.util.fix_length(y, size=int(SAMPLE_RATE * 3))

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    lengths.append(mel_db.shape[1])

print("Unique lengths:", set(lengths))

Unique lengths: {94}


In [12]:
SAVE_DIR = "features_mel"
os.makedirs(SAVE_DIR, exist_ok=True)

for file in os.listdir(AUDIO_DIR):
    if file.endswith(".wav"):
        path = os.path.join(AUDIO_DIR, file)

        y, sr = librosa.load(path, sr=SAMPLE_RATE)
        y = librosa.util.fix_length(y, size=int(SAMPLE_RATE * 3))

        mel = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=N_MELS
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        # normalize
        mel_db = (mel_db - np.mean(mel_db)) / np.std(mel_db)

        save_path = os.path.join(SAVE_DIR, file.replace(".wav", ".npy"))
        np.save(save_path, mel_db)

print("Done.")

Done.


In [13]:
import os
import numpy as np
from collections import defaultdict

FEATURE_DIR = "features_mfcc"
SAVE_DIR = "features_mfcc_norm"

os.makedirs(SAVE_DIR, exist_ok=True)

# group files by speaker
speaker_files = defaultdict(list)

for file in os.listdir(FEATURE_DIR):
    if file.endswith(".npy"):
        speaker_id = file.split("-")[-1].replace(".npy", "")
        speaker_files[speaker_id].append(file)

# compute stats + normalize
for speaker, files in speaker_files.items():
    all_feats = []

    # stack all features for this speaker
    for file in files:
        data = np.load(os.path.join(FEATURE_DIR, file))
        all_feats.append(data)

    all_feats = np.hstack(all_feats)

    # per-channel stats (correct way)
    mean = np.mean(all_feats, axis=1, keepdims=True)
    std = np.std(all_feats, axis=1, keepdims=True) + 1e-8

    # normalize each file
    for file in files:
        data = np.load(os.path.join(FEATURE_DIR, file))
        norm = (data - mean) / std

        save_path = os.path.join(SAVE_DIR, file)
        np.save(save_path, norm)

print("MFCC normalization done.")

MFCC normalization done.
